# 02 - LimiX-2 scoring and explanations

This notebook fits LimiX-2, LightGBM and logistic regression on the same synthetic cash-flow
portfolio, compares ranking and calibration, and explains individual decisions.

> **Licence:** LimiX-2 weights are non-commercial. The notebook detects whether the checkpoint is
> available and falls back to the tree and linear baselines otherwise. See `docs/limix.md`.


In [1]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split

from opencashflow.data.synthetic import generate_portfolio
from opencashflow.features.cashflow_features import FeatureConfig
from opencashflow.models import (
    LimiXScorer,
    LogisticBaseline,
    TreeBaseline,
    calibration_table,
    evaluate_binary,
    explain_model,
)

pd.set_option("display.width", 140)

## 1. Build a labelled portfolio

We use 120 companies and 18 months of observable history. The train/test split is stratified so the default rate is similar in both sets.

In [2]:
portfolio = generate_portfolio(
    n_companies=120,
    months=18,
    default_rate=0.15,
    seed=7,
)
config = FeatureConfig(window_days=18 * 31, as_of=portfolio.metadata["observation_end"])
X, y = portfolio.to_features(config)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
print(portfolio.summary())
print("Train:", X_train.shape, "Test:", X_test.shape, "Default rate:", round(float(y.mean()), 3))

120 companies, 63,410 transactions, default rate 15.0%, 2023-01-02 to 2024-06-29
Train: (84, 151) Test: (36, 151) Default rate: 0.15


## 2. Check LimiX availability

The wrapper resolves the checkpoint from `LIMIX_MODEL_PATH` and the configs from `LIMIX_SRC` / `LIMIX_CONFIG_DIR`. No weights are downloaded automatically.

In [3]:
limix_info = LimiXScorer.availability()
print("available:", limix_info["available"])
print("reasons:", limix_info["reasons"])
print("checkpoint:", Path(limix_info["model_path"]).name if limix_info["model_path"] else None)
print("source:", Path(limix_info["limix_src"]).name if limix_info["limix_src"] else None)

available: True
reasons: []
checkpoint: LimiX-2-1016.ckpt
source: limix_src


## 3. Fit the models

LimiX uses in-context learning: `fit` stores the labelled reference set and `predict` runs the frozen checkpoint. One pipeline is used on CPU.

In [4]:
models = {
    "Logistic": LogisticBaseline(),
    "LightGBM": TreeBaseline("lightgbm", n_estimators=300),
}
if limix_info["available"]:
    models["LimiX-2 (1 pipeline)"] = LimiXScorer(n_pipelines=1, seed=0, verbose=False)
else:
    print("LimiX unavailable; using tree and linear baselines only.")

scores = {}
rows = []
for name, model in models.items():
    model.fit(X_train, y_train)
    probabilities = model.predict_pd(X_test)
    scores[name] = probabilities
    metrics = evaluate_binary(y_test, probabilities)
    rows.append(
        {
            "model": name,
            "roc_auc": metrics["roc_auc"],
            "ks": metrics["ks"],
            "brier": metrics["brier"],
            "log_loss": metrics["log_loss"],
        }
    )

metrics_frame = pd.DataFrame(rows).sort_values("roc_auc", ascending=False).reset_index(drop=True)
metrics_frame.round(3)

arch_version=2.0 -> inference.v2_0.predictor (inference.v2_0.predictor)
Mixed precision is not supported for CPU inference, so it has been automatically disabled
use limix v2.0


,model,roc_auc,ks,brier,log_loss
0,LightGBM,0.755,0.535,0.107,0.369
1,LimiX-2 (1 pipeline),0.716,0.503,0.143,0.441
2,Logistic,0.523,0.342,0.227,1.053


## 4. Ranking and calibration

AUC measures ranking. Brier score and log loss also penalise overconfident probabilities, which matters for pricing and provisioning.

In [5]:
fig, ax = plt.subplots(figsize=(7, 5))
for name, probabilities in scores.items():
    fpr, tpr, _ = roc_curve(y_test, probabilities)
    ax.plot(fpr, tpr, label=f"{name} (AUC={roc_auc_score(y_test, probabilities):.3f})")
ax.plot([0, 1], [0, 1], linestyle="--", color="grey", linewidth=1)
ax.set_xlabel("False positive rate")
ax.set_ylabel("True positive rate")
ax.set_title("ROC curves on the synthetic holdout")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

In [6]:
best_name = metrics_frame.iloc[0]["model"]
best_probabilities = scores[best_name]
calibration = calibration_table(y_test, best_probabilities, n_bins=8)

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot([0, 1], [0, 1], linestyle="--", color="grey", linewidth=1, label="Perfect calibration")
ax.plot(calibration["mean_predicted"], calibration["observed"], marker="o", label=best_name)
ax.set_xlabel("Mean predicted PD")
ax.set_ylabel("Observed default rate")
ax.set_title(f"Calibration: {best_name}")
ax.legend()
plt.tight_layout()
plt.show()

calibration.round(3)

,bin_lower,bin_upper,count,mean_predicted,observed,gap
0,-0.001,0.125,25,0.030,0.080,0.050
1,0.125,0.250,4,0.187,0.000,-0.187
2,0.250,0.375,1,0.272,0.000,-0.272
3,0.375,0.500,3,0.442,0.667,0.225
4,0.500,0.625,1,0.579,0.000,-0.579
5,0.625,0.750,2,0.681,0.500,-0.181


## 5. Explain the riskiest application

Tree models use exact TreeSHAP. LimiX uses occlusion: replace one feature at a time with a reference value and measure the PD change. Reason codes are model-relative, not causal.

In [7]:
best_model = models[best_name]
order = np.argsort(-best_probabilities)[:3]
selected = X_test.iloc[order]
explanations = explain_model(best_model, selected, max_features=25)

for explanation in explanations:
    print(f"\nEntity {explanation.entity_id} | PD={explanation.prediction:.3f} | method={explanation.method}")
    for reason in explanation.reason_codes(4):
        print("  -", reason)


Entity SME0069 | PD=0.723 | method=tree_shap
  - Heuristic stability score (0-100, higher is better) (cashflow_stability_score) increased predicted risk.
  - Loan repayments divided by revenue (debt_service_ratio) increased predicted risk.
  - Standard deviation of month-on-month revenue growth (revenue_growth_volatility) increased predicted risk.
  - Share of transactions booked in the last three days of a month (month_end_ratio) increased predicted risk.

Entity SME0007 | PD=0.639 | method=tree_shap
  - Heuristic stability score (0-100, higher is better) (cashflow_stability_score) increased predicted risk.
  - Loan repayments divided by revenue (debt_service_ratio) increased predicted risk.
  - Consistency of the payroll day of month (0-1) (salary_day_consistency) increased predicted risk.
  - Standard deviation of month-on-month revenue growth (revenue_growth_volatility) increased predicted risk.

Entity SME0108 | PD=0.579 | method=tree_shap
  - Heuristic stability score (0-100, hi

In [8]:
explanation = explanations[0]
explanation.top(8)[["feature", "value", "contribution", "direction", "description"]]

,feature,value,contribution,direction,description
0,cashflow_stability_score,71.638653,1.560069,increases risk,"Heuristic stability score (0-100, higher is be..."
1,debt_service_ratio,0.067682,0.767808,increases risk,Loan repayments divided by revenue.
2,revenue_growth_volatility,0.377622,0.537805,increases risk,Standard deviation of month-on-month revenue g...
3,month_end_ratio,0.020635,0.416900,increases risk,Share of transactions booked in the last three...
4,revenue_growth_3m,0.227884,0.360995,increases risk,Last 3 months revenue versus the prior 3 months.
5,bank_fees_to_revenue,0.001403,0.312381,increases risk,Bank fees divided by revenue.
6,outflow_quality,0.917543,0.270049,increases risk,Operating expenses as a share of all outflows.
7,leverage_health_score,54.585471,0.258888,increases risk,"Heuristic leverage score (0-100, higher is bet..."


## 6. What to remember

* On this 120-company portfolio's 36-row holdout LightGBM ranks first with ROC-AUC 0.755, ahead of LimiX-2 at 0.716 and logistic regression at 0.523.
* Brier score and log loss follow the same order: LightGBM 0.107 and 0.369, LimiX-2 0.143 and 0.441, logistic 0.227 and 1.053.
* Calibration is shown for LightGBM, the top-ranked model. Several bins have observed rates away from their mean prediction.
* Explanations use TreeSHAP on LightGBM. Reason codes are model-relative statements, not causal claims or regulated adverse-action codes. One CPU pipeline is a demo, not a leaderboard result.

In [9]:
for model in models.values():
    if hasattr(model, "close"):
        model.close()
print("Released LimiX predictor memory.")

Released LimiX predictor memory.
